# IIO MCA Backend Walkthrough (vdpp_pulse_processor / vdpp_input_filter)

Live-tested calling pattern for `IIODigitizerBackend`'s `MCABackend` half,
newly implemented against the board's `vdpp-pulse-processor.c` (signal
parameters, acquisition control, statistics, filters, pulse memory,
histogram) and `vdpp-input-filter.c` (FIR/IIR low-pass filter, temperature
compensation) drivers -- see `mca-architecture.md` for how the two cores
relate.

**This live run caught and fixed a real bug.** An earlier revision of
`iio_backend.py` reimplemented the driver's `PP_FMT_X2`/`PP_FMT_X8`/
`PP_FMT_PLUS1_X8` conversions (`pp_field_to_user()`/`pp_field_from_user()`)
client-side, on the assumption that the sysfs attribute held the raw
register value. It doesn't: `pp_field_show()`/`pp_field_store()` already
apply that conversion *inside the kernel*, so the sysfs value already is
the physical/user value -- confirmed live by writing `"24"` directly to
`pretrigger_samples` and reading back `"24"`, not `"12"`. The old code was
applying the same conversion a second time, which silently wrote a
roughly-halved-or-eighthed value to hardware on every affected field while
still round-tripping correctly through `get_*()` (a symmetric bug, invisible
to a get-after-set check) -- it only surfaced because a double-converted
`crrc2_fdelay` value happened to fall outside that field's valid range,
raising a real `OSError`. Every affected field below is now a plain
passthrough and has been re-confirmed live, including at both ends of its
valid range.

**One confirmed, currently-open gap:** `read_histogram()`/
`read_waveform_banks()` are implemented per the driver source but currently
raise `RuntimeError` on every call against this board. Their
`bin_attribute` sysfs files (`debug_data`, `histogram_data`) are not
discoverable through the IIO network transport at all here (checked via
`iio._d_attr_count()`/`_d_debug_attr_count()`/`_d_buffer_attr_count()`, all
three exclude them -- consistent with their absence from `iio_info`'s
device-attribute dump). Every *other* attribute on the same device, and
even an oversized *text* attribute (`vdpp_scope`'s `viewer_data`) over the
same transport, works fine -- so this looks like a driver/`iiod`
`bin_attribute` discovery gap specific to this board's firmware+`iiod`
build, not a client-side bug. `MCAWorker` tolerates it (falls back to the
last-known histogram/waveforms, logged once rather than every tick), which
is what lets the rest of this notebook's live measurement loop keep working
below.

**Known, by-design gaps** (see `iio_backend.py`'s module docstring for the
full reasoning):
- `d.mca.sync.*` -- no sync-trigger core found in either driver source;
  raises `NotImplementedError`. Unused by `MCAController`, so this is safe.
- `d.mca.statistics.get_events_lost()` -- no matching register in this HLS
  core (IP version 101); raises `NotImplementedError`. `MCAWorker` already
  tolerates this (falls back to 0) so the live GUI readback loop doesn't
  break on it.
- `d.mca.clear_spectrum()` -- no histogram-clear register either;
  `mca-architecture.md` says this driver revision deliberately removed the
  legacy implicit-stop/start clear, so this raises `NotImplementedError`
  rather than silently resetting elapsed-time/statistics as a side effect.
- `d.mca.get_edge_det_coeff()`/`set_edge_det_coeff()` -- no register in
  either driver; this is a software-only shadow value with no hardware
  effect, kept only so `MCAController`'s unconditional default-hydration
  pass doesn't crash on construction.
- `d.mca.get_dma_enable()`/`set_dma_enable()` -- list-mode DMA is owned by
  a separate `vdpp-lm-frame` driver that hasn't been ported to this backend
  yet (`d.mca_dma` is `None`). The getter reads the real, read-only
  `list_buffer_active` diagnostic (confirmed present in `iio_info`'s dump,
  alongside `vdpp_lm_frame` itself); the setter is a documented no-op.

**Hardware note:** run `iio_info -u "ip:<host>:30431"` first and confirm
both `vdpp_pulse_processor` and `vdpp_input_filter` show up (one instance
each per scope channel) before assuming this notebook applies -- see
`scope_walkthrough.ipynb`'s own hardware note about the board's IP/driver
revision moving around.

In [1]:
import sys
sys.path.insert(0, "../src")

import time

import numpy as np
import matplotlib.pyplot as plt

from nlab.hardware.digitizer import Digitizer

HOST = "192.168.10.128"
PORT = 30431
CHANNEL = 0  # 0 -> first vdpp_pulse_processor/vdpp_input_filter pair found,
             # 1 -> second (same discovery-order convention as vdpp_scope)

d = Digitizer.from_iio(CHANNEL, f"ip:{HOST}:{PORT}")
mca = d.mca

print("connected            :", mca is not None)
print("mca_available()      :", d.mca_available())
print("mca_hardware_present():", d._backend.mca_hardware_present())

if not d.mca_available():
    raise RuntimeError(
        "no vdpp_pulse_processor/vdpp_input_filter device found for this "
        "channel -- check `iio_info -u ...` output before continuing"
    )

IIO backend ch0: same-name devices are selected by sorted IIO probe index; user-api.md says this is not a guaranteed physical A/B identity because remote IIO exposes no platform path for authoritative pairing


connected            : True
mca_available()      : True
mca_hardware_present(): True


## Device info

`hw_version`/`sw_version`/`id_number` are plain read-only registers.
`vdpp-pulse-processor.c`'s own `pp_probe()` refuses to bind unless
`hw_version` reads exactly `101` (`PP_HW_VERSION_EXPECTED`), so seeing that
value here is a genuine identity check, the same way `scope.get_ip_version()`
is in the scope notebook.

In [2]:
print("hw_version :", mca.get_hw_version(), "(expect 101)")
print("sw_version :", mca.get_sw_version())
print("id_number  :", mca.get_id_number())

hw_version : 101 (expect 101)
sw_version : 103013
id_number  : 1


## Signal parameters

`trigger_level`, `pulse_polarity`, `baseline_window`, `pretrigger_samples`,
`frame_samples` and `trg_source` are all registers on `vdpp_pulse_processor`
-- see `MCA_PARAMETER_SPECS` in `mca.py` for the valid ranges
`MultiChannelAnalyzer` validates client-side before ever reaching the
backend.

`pretrigger_samples`/`frame_samples` are `PP_FMT_X2` fields in the driver,
but the kernel's own `pp_field_show()`/`pp_field_store()` already convert
between the raw register and the physical sample count -- the sysfs value
(and therefore `get_*`/`set_*` here) is already in samples, confirmed live
(see the intro's account of the double-conversion bug this notebook's run
caught and fixed).

`edge_det_coeff` has **no matching register** in either driver source (see
the intro) -- it's a software-only shadow on this backend, so it round-trips
correctly here but has zero effect on the hardware. Written first, before
anything else, since `MCAController._send_defaults()` would otherwise be the
only thing exercising it.

In [3]:
mca.set_edge_det_coeff(1)
print("edge_det_coeff (software-only, no HW effect):", mca.get_edge_det_coeff())

mca.set_trigger_level(-512)
mca.set_pulse_polarity(0)
mca.set_baseline_window(3)
mca.set_pretrigger_samples(24)
mca.set_frame_samples(256)
mca.set_trg_source(0)
mca.set_ext_trig_enable(False)

print("trigger_level      :", mca.get_trigger_level())
print("pulse_polarity     :", mca.get_pulse_polarity())
print("baseline_window    :", mca.get_baseline_window())
print("pretrigger_samples :", mca.get_pretrigger_samples())
print("frame_samples      :", mca.get_frame_samples())
print("trg_source         :", mca.get_trg_source())
print("ext_trig_enable    :", mca.get_ext_trig_enable())

edge_det_coeff (software-only, no HW effect): 1
trigger_level      : -512
pulse_polarity     : 0
baseline_window    : 3
pretrigger_samples : 24
frame_samples      : 256
trg_source         : 0
ext_trig_enable    : False


## FIR/IIR low-pass filter and temperature compensation (`vdpp_input_filter`)

These live on a *different* IIO device (`vdpp_input_filter`) than the rest
of this notebook (`vdpp_pulse_processor`) -- see `mca-architecture.md`'s
pipeline diagram. `set_preset()` picks one of three factory FIR presets
(`vdpp_input_filter_presets[]` in the driver: index 0 = `"200mhz"`, 1 =
`"70mhz"`, 2 = `"moving_average"` -- note `mca.py`'s own comment mislabels
index 1 as "700 MHz"); `get_coeffs()` reads back the 12 signed Q1.15
coefficients each preset just wrote, confirming both that the write landed
and that the presets are what they claim to be (index 2's coefficients are
all identical, `2730` -- a genuine moving average, not e.g. an accidental
duplicate of index 0 or 1).

`get_iir_average()` reads the IIR filter's live output through a standard
IIO channel attribute (`in_voltage0_raw`), not a plain device attribute --
polled a few times below to confirm it's a real, moving ADC-derived value,
not stuck at a constant.

`temp_coeff`/`temp_offset` round-trip through a raw register plus a
`*_scale` attribute read live off the device (`temperature_coefficient_scale`
= 2^-15) -- the value written (`-0.0001`) doesn't come back exactly (it's
quantized to the nearest multiple of 2^-15), which is expected, not a bug.

In [4]:
mca.filters.lp.set_preset(0)  # "200mhz"
print("lp coeffs size            :", mca.filters.lp.get_size(), "(expect 12)")
print("lp coeffs (200mhz preset) :", mca.filters.lp.get_coeffs())

mca.filters.lp.set_preset(1)  # "70mhz"
print("lp coeffs (70mhz preset)  :", mca.filters.lp.get_coeffs())

mca.filters.lp.set_preset(2)  # "moving_average"
print("lp coeffs (moving_average):", mca.filters.lp.get_coeffs())

mca.filters.lp.set_preset(0)  # restore

for _ in range(3):
    print("iir_lp_average :", mca.filters.lp.get_iir_average())
    time.sleep(0.05)

orig_temp_coeff = mca.get_temp_coeff()
orig_temp_offset = mca.get_temp_offset()
print("temp_coeff (current) :", orig_temp_coeff)
print("temp_offset (current):", orig_temp_offset)

mca.set_temp_coeff(-0.0001)
mca.set_temp_offset(5)
print("temp_coeff (set, quantized to nearest 2^-15):", mca.get_temp_coeff())
print("temp_offset (set)                           :", mca.get_temp_offset())

mca.set_temp_coeff(orig_temp_coeff)
mca.set_temp_offset(orig_temp_offset)
print("temp_coeff (restored) :", mca.get_temp_coeff())
print("temp_offset (restored):", mca.get_temp_offset())

lp coeffs size            : 12 (expect 12)
lp coeffs (200mhz preset) : [   -2   148  -677  2050 -5434 20306 20306 -5434  2050  -677   148    -2]
lp coeffs (70mhz preset)  : [ -167  -661  -724  1406  6120 10401 10401  6120  1406  -724  -661  -167]
lp coeffs (moving_average): [2730 2730 2730 2730 2730 2730 2730 2730 2730 2730 2730 2730]
iir_lp_average : 12580
iir_lp_average : 12582
iir_lp_average : 12569
temp_coeff (current) : 0.0
temp_offset (current): 0
temp_coeff (set, quantized to nearest 2^-15): -9.1552734375e-05
temp_offset (set)                           : 5
temp_coeff (restored) : 0.0
temp_offset (restored): 0


## CRRC2 and trapezoidal shaping filters

**Two things confirmed live here, not just derived from reading the source:**

1. `crrc2.get_Fdelay()`/`trapezoid.get_R()` (`PP_FMT_PLUS1_X8`/
   `PP_FMT_TRAPEZOID_R`, `user = (raw + 1) * 8` in the driver) and
   `trapezoid.get_M()`/`get_E()` (`PP_FMT_X8`) are, like the signal
   parameters above, already converted by the kernel -- plain passthrough
   here, tested at both ends of each field's valid range below (`crrc2_
   fdelay`: 8 and 1016; `trapezoid_r`: 16 and 4088).
2. `trapezoid.get_T()`/`set_T()` maps to the driver's `trapezoid_beta_raw`
   attribute (raw, unscaled uint32 -- the legacy pole-zero-compensation
   value computed as `round(exp(-8 / T) * 2**31)` per `mca-architecture.md`),
   **not** the driver's own `trapezoid_time` attribute despite the name.
   `trapezoid.get_E()`/`set_E()` maps to `trapezoid_time` instead. This was
   worked out from `mca.py`'s `MCA_PARAMETER_SPECS` numeric ranges
   (`TRAPEZ_T`: full uint32 range; `TRAPEZ_E`: 0..16376 step 8) against the
   driver's `pp_fields[]` table -- the round-trips below confirm both
   registers accept their respective values without error.

Also confirmed live: `trapezoid_r`'s *real* minimum is `16`, not the `0`
`mca.py`'s stale `RangeSpec` for `MCAParam.TRAPEZ_R` allows --
`set_R(8)` raises a raw `OSError` (`ERANGE`), exactly as
`mca-architecture.md` says ("the legacy expression is undefined for 8").

In [5]:
mca.filters.crrc2.set_Cdelay(8)
print("crrc2 Cdelay   :", mca.filters.crrc2.get_Cdelay())

mca.filters.crrc2.set_Fdelay(8)      # field minimum
print("crrc2 Fdelay (min)   :", mca.filters.crrc2.get_Fdelay())
mca.filters.crrc2.set_Fdelay(1016)   # field maximum
print("crrc2 Fdelay (max)   :", mca.filters.crrc2.get_Fdelay())

mca.filters.crrc2.set_pzc_coeff(32761)
print("crrc2 pzc_coeff:", mca.filters.crrc2.get_pzc_coeff())

mca.filters.trapezoid.set_enable(False)

mca.filters.trapezoid.set_R(16)      # driver's real minimum, not mca.py's stale 0
print("trapez R (min=16)    :", mca.filters.trapezoid.get_R())
mca.filters.trapezoid.set_R(4088)    # field maximum
print("trapez R (max=4088)  :", mca.filters.trapezoid.get_R())
mca.filters.trapezoid.set_R(64)      # back to a normal operating value

try:
    mca.filters.trapezoid.set_R(8)
    print("trapez R (8): unexpectedly SUCCEEDED -- driver min has changed!")
except OSError as e:
    print(f"trapez R (8): raised OSError as expected (below real min 16): {e}")

mca.filters.trapezoid.set_M(64)
mca.filters.trapezoid.set_T(96)     # -> trapezoid_beta_raw, see markdown above
mca.filters.trapezoid.set_E(256)    # -> trapezoid_time, see markdown above
mca.filters.trapezoid.set_FT(0)

print("trapez enable :", mca.filters.trapezoid.get_enable())
print("trapez R      :", mca.filters.trapezoid.get_R())
print("trapez M      :", mca.filters.trapezoid.get_M())
print("trapez T      :", mca.filters.trapezoid.get_T())
print("trapez E      :", mca.filters.trapezoid.get_E())
print("trapez FT     :", mca.filters.trapezoid.get_FT())

crrc2 Cdelay   : 8
crrc2 Fdelay (min)   : 8
crrc2 Fdelay (max)   : 1016
crrc2 pzc_coeff: 32761
trapez R (min=16)    : 16
trapez R (max=4088)  : 4088
trapez R (8): raised OSError as expected (below real min 16): [Errno 34] Result too large
trapez enable : False
trapez R      : 64
trapez M      : 64
trapez T      : 96
trapez E      : 256
trapez FT     : 0


## CFD, charge comparison, and PSD zero-crossing

`cfd.get_factor()`/`set_factor()` round-trips through `cfd_factor_raw` (a
plain, *unscaled* uint16 register -- `PP_FMT_RAW`, the one format the
kernel does *not* convert, unlike X2/X8/PLUS1_X8 above) and the separate
read-only `cfd_factor_scale` attribute (2^-15) -- `0.4` comes back
quantized (`0.399993...`), same reasoning as `temp_coeff` above.
`cfd.get_delay()` and both the CFD and PSD time-window pairs are
`PP_FMT_X2`, already converted by the kernel like `pretrigger_samples`.

In [6]:
mca.filters.cfd.set_enable(False)
mca.filters.cfd.set_factor(0.4)
mca.filters.cfd.set_delay(2)
mca.filters.cfd.set_time_window_low(8)
mca.filters.cfd.set_time_window_high(64)
print("cfd enable    :", mca.filters.cfd.get_enable())
print("cfd factor    :", mca.filters.cfd.get_factor(), "(quantized to nearest 2^-15, not exactly 0.4)")
print("cfd delay     :", mca.filters.cfd.get_delay())
print("cfd tw_low    :", mca.filters.cfd.get_time_window_low())
print("cfd tw_high   :", mca.filters.cfd.get_time_window_high())

mca.filters.charge_comparison.set_enable(False)
mca.filters.charge_comparison.set_time(8)
print("cc enable     :", mca.filters.charge_comparison.get_enable())
print("cc time       :", mca.filters.charge_comparison.get_time())

mca.filters.psd_zc.set_enable(False)
mca.filters.psd_zc.set_mode(0)
mca.filters.psd_zc.set_time_window_low(8)
mca.filters.psd_zc.set_time_window_high(16)
print("psd_zc enable :", mca.filters.psd_zc.get_enable())
print("psd_zc mode   :", mca.filters.psd_zc.get_mode())
print("psd_zc tw_low :", mca.filters.psd_zc.get_time_window_low())
print("psd_zc tw_high:", mca.filters.psd_zc.get_time_window_high())

cfd enable    : False
cfd factor    : 0.399993896484375 (quantized to nearest 2^-15, not exactly 0.4)
cfd delay     : 2
cfd tw_low    : 8
cfd tw_high   : 64
cc enable     : False
cc time       : 8
psd_zc enable : False
psd_zc mode   : 0
psd_zc tw_low : 8
psd_zc tw_high: 16


## Pulse-memory signal routing, energy binning, and the DMA diagnostic

`mem1_sig_select`/`mem2_sig_select` choose which internal debug signal each
2048-sample pulse memory captures (0=input, 1=trigger, 2=trapezoid,
3=trapezoid-energy, 4=cfd, 5=cfd-window, 6=psd-zc-window, 7=logic-trigger --
`vdpp-pulse-processor.c`'s `pp_debug_signal_available_show()`). Set to
different values here on purpose; would normally make the two waveform-bank
plots later in this notebook visibly different signals -- see the intro for
why those plots are currently empty instead (the confirmed `bin_attribute`
discovery gap).

`get_dma_enable()` reads the real, read-only `list_buffer_active` gate;
`set_dma_enable()` is a documented no-op (list-mode DMA belongs to the
not-yet-ported `vdpp-lm-frame` driver) -- confirmed below that it doesn't
raise, and that `get_dma_enable()` doesn't change as a result of calling it.

In [7]:
mca.set_mem1_sig_select(0)  # input
mca.set_mem2_sig_select(1)  # trigger
print("mem1_sig_select:", mca.get_mem1_sig_select())
print("mem2_sig_select:", mca.get_mem2_sig_select())
print("mem_amount     :", mca.get_mem_amount(), "(expect 2)")
print("mem_frame_sizes:", mca.get_mem_frame_sizes(), "(expect [2048 2048])")

mca.set_energy_bin(0)
mca.set_pileup_window(0)
print("energy_bin     :", mca.get_energy_bin())
print("pileup_window  :", mca.get_pileup_window())

print("dma_enable (list_buffer_active, before)                    :", mca.get_dma_enable())
mca.set_dma_enable(True)  # documented no-op -- see markdown above
print("dma_enable (list_buffer_active, after set_dma_enable(True)):", mca.get_dma_enable())

print("histogram_size :", mca.get_histogram_size(), "(expect 16384, fixed constant, no register)")

mem1_sig_select: 0
mem2_sig_select: 1
mem_amount     : 2 (expect 2)
mem_frame_sizes: [2048 2048] (expect [2048 2048])
energy_bin     : 0
pileup_window  : 0
dma_enable (list_buffer_active, before)                    : False
dma_enable (list_buffer_active, after set_dma_enable(True)): False
histogram_size : 16384 (expect 16384, fixed constant, no register)


## Run a short measurement, polling waveforms + statistics live

Mirrors `MCAController._start_polling_only()`: set a time limit, call
`mca.start()` (`global_enable=1`), then poll. `get_measurement_in_progress()`
and every statistic have no run-state restriction and show live-changing
values on each iteration (`count_rate` genuinely moves between prints
below); `get_measurement_in_progress()` reads `True` throughout and drops
to `False` on its own once `time_limit` elapses.

`acquire_spectrum()`/`acquire_waveforms()` are also called here, each
wrapped in `try`/`except` -- confirmed live, both currently raise on every
call (the `bin_attribute` discovery gap from the intro), not just while
running. `MultiChannelAnalyzer`'s statistics/measurement-state calls have
no dependency on either of them, which is exactly what keeps this loop
useful despite that gap.

In [8]:
mca.set_time_limit(3)  # seconds
mca.start()
print("measurement_in_progress right after start():", mca.get_measurement_in_progress())

t0 = time.monotonic()
n_iter = 0
histogram_ever_ok = False
waveforms_ever_ok = False
last_debug1 = last_debug2 = None
last_histogram_error = last_waveform_error = None

try:
    while mca.get_measurement_in_progress() and time.monotonic() - t0 < 10.0:
        stats = mca.statistics

        try:
            mca.acquire_spectrum()
            histogram_ever_ok = True
            histogram_note = "OK"
        except (OSError, RuntimeError) as e:
            last_histogram_error = e
            histogram_note = f"{type(e).__name__} (see intro)"

        try:
            last_debug1, last_debug2 = mca.acquire_waveforms()
            waveforms_ever_ok = True
            waveform_note = "OK"
        except (OSError, RuntimeError) as e:
            last_waveform_error = e
            waveform_note = f"{type(e).__name__} (see intro)"

        n_iter += 1
        if n_iter <= 3 or n_iter % 5 == 0:
            print(
                f"t={time.monotonic() - t0:4.1f}s  "
                f"count_rate={stats.get_count_rate():>7}  "
                f"elapsed={stats.get_elapsed_time():>4} (0.1s ticks)  "
                f"deadtime={stats.get_pulse_deadtime():>6}  "
                f"acquire_spectrum(): {histogram_note:32s}  "
                f"acquire_waveforms(): {waveform_note}"
            )
        time.sleep(0.3)
finally:
    mca.stop()

print(f"\ntotal iterations           : {n_iter}")
print(f"acquire_spectrum() ever OK : {histogram_ever_ok}"
      + ("" if histogram_ever_ok else f"  (last error: {last_histogram_error})"))
print(f"acquire_waveforms() ever OK: {waveforms_ever_ok}"
      + ("" if waveforms_ever_ok else f"  (last error: {last_waveform_error})"))
print("measurement_in_progress after stop():", mca.get_measurement_in_progress())

measurement_in_progress right after start(): True
t= 0.0s  count_rate= 142552  elapsed=   0 (0.1s ticks)  deadtime=     0  acquire_spectrum(): RuntimeError (see intro)          acquire_waveforms(): RuntimeError (see intro)


t= 0.3s  count_rate= 142552  elapsed=   4 (0.1s ticks)  deadtime=    17  acquire_spectrum(): RuntimeError (see intro)          acquire_waveforms(): RuntimeError (see intro)


t= 0.6s  count_rate= 142552  elapsed=   8 (0.1s ticks)  deadtime=    35  acquire_spectrum(): RuntimeError (see intro)          acquire_waveforms(): RuntimeError (see intro)


t= 1.2s  count_rate= 142765  elapsed=  16 (0.1s ticks)  deadtime=    70  acquire_spectrum(): RuntimeError (see intro)          acquire_waveforms(): RuntimeError (see intro)


t= 2.7s  count_rate= 142891  elapsed=  36 (0.1s ticks)  deadtime=   158  acquire_spectrum(): RuntimeError (see intro)          acquire_waveforms(): RuntimeError (see intro)



total iterations           : 10
acquire_spectrum() ever OK : False  (last error: 'histogram_data' is not reachable through this IIO network context -- confirmed live against a real board: it does not appear in vdpp_pulse_processor's regular, debug, or buffer attribute lists at all (checked via iio._d_attr_count()/_d_debug_attr_count()/_d_buffer_attr_count(), all three exclude it, matching its absence from `iio_info`'s device-attribute dump), so iiod rejects the read by name before it ever reaches vdpp-pulse-processor.c's pp_debug_snapshot_read()/pp_histogram_snapshot_read(). This looks like a driver/iiod bin_attribute discovery gap on this board's firmware+iiod build (device_attribute-based attributes on the same device, and even the oversized text device_attribute viewer_data on vdpp_scope, work fine over this same transport) -- not something a bigger read buffer or a client-side retry can fix.)
acquire_waveforms() ever OK: False  (last error: 'debug_data' is not reachable through th

## Plot the waveform banks captured above, if any

Only plots if `acquire_waveforms()` succeeded at least once in the loop
above -- given the confirmed `bin_attribute` gap from the intro, expect
this to report "not captured" rather than show a plot right now. Once that
gap is fixed board-side, this should show two visibly different curves
(`mem1_sig_select=0`/input vs `mem2_sig_select=1`/trigger, set earlier).

In [9]:
if last_debug1 is None:
    print(
        "No waveform bank was ever captured this run -- consistent with "
        "the confirmed bin_attribute discovery gap from the intro, not a "
        "new problem. Skipping the plot."
    )
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
    axes[0].plot(last_debug1)
    axes[0].set_title(f"pulse memory 1 (mem1_sig_select={mca.get_mem1_sig_select()})")
    axes[1].plot(last_debug2)
    axes[1].set_title(f"pulse memory 2 (mem2_sig_select={mca.get_mem2_sig_select()})")
    for ax in axes:
        ax.set_xlabel("sample")
        ax.set_ylabel("ADC counts (int16)")
    plt.tight_layout()
    plt.show()

No waveform bank was ever captured this run -- consistent with the confirmed bin_attribute discovery gap from the intro, not a new problem. Skipping the plot.


## Read the histogram now that the measurement is stopped

Per `pp_histogram_snapshot_read()`, this is expected to succeed now
(`enable=0`) where the identical call raised in the loop above *if* the
`bin_attribute` discovery gap were the only issue while running -- but
since that gap blocks the read unconditionally (not just while running,
see the intro), expect this to raise the same way here too, and for this
cell to report that rather than plot anything.

In [10]:
try:
    histogram = mca.acquire_spectrum()
    print("histogram shape/dtype:", histogram.shape, histogram.dtype)
    assert histogram.shape == (mca.get_histogram_size(),), "histogram length != get_histogram_size()"

    plt.figure(figsize=(10, 4))
    plt.plot(histogram)
    plt.xlabel("energy channel")
    plt.ylabel("counts")
    plt.title(f"vdpp_pulse_processor ch{CHANNEL} accumulated spectrum")
    plt.tight_layout()
    plt.show()
except (OSError, RuntimeError) as e:
    print(f"acquire_spectrum() raised {type(e).__name__} even while stopped -- "
          f"consistent with the confirmed bin_attribute discovery gap from the "
          f"intro (not run-state related). Skipping the plot.\n\n{e}")

acquire_spectrum() raised RuntimeError even while stopped -- consistent with the confirmed bin_attribute discovery gap from the intro (not run-state related). Skipping the plot.

'histogram_data' is not reachable through this IIO network context -- confirmed live against a real board: it does not appear in vdpp_pulse_processor's regular, debug, or buffer attribute lists at all (checked via iio._d_attr_count()/_d_debug_attr_count()/_d_buffer_attr_count(), all three exclude it, matching its absence from `iio_info`'s device-attribute dump), so iiod rejects the read by name before it ever reaches vdpp-pulse-processor.c's pp_debug_snapshot_read()/pp_histogram_snapshot_read(). This looks like a driver/iiod bin_attribute discovery gap on this board's firmware+iiod build (device_attribute-based attributes on the same device, and even the oversized text device_attribute viewer_data on vdpp_scope, work fine over this same transport) -- not something a bigger read buffer or a client-side retry ca

## Confirming the documented gaps actually raise, not silently misbehave

Each call below is expected to raise `NotImplementedError` -- if any of
them *doesn't* raise (or raises something else), that's new information
this notebook's intro and `iio_backend.py`'s module docstring both need
updating for, not something to quietly work around here.

In [11]:
def expect_not_implemented(label, fn):
    try:
        fn()
        print(f"{label}: unexpectedly SUCCEEDED (update the notes!)")
    except NotImplementedError as e:
        print(f"{label}: raised NotImplementedError as expected ({e})")
    except Exception as e:
        print(f"{label}: raised {type(e).__name__} instead of NotImplementedError: {e}")

expect_not_implemented("clear_spectrum()", mca.clear_spectrum)
expect_not_implemented("statistics.get_events_lost()", mca.statistics.get_events_lost)
expect_not_implemented("sync.get_enable()", mca.sync.get_enable)
expect_not_implemented("sync.get_timestamp()", mca.sync.get_timestamp)

clear_spectrum(): raised NotImplementedError as expected (IIO MCA backend: vdpp-pulse-processor.c has no histogram-clear register. Per mca-architecture.md, this driver revision deliberately replaced the legacy library's implicit stop/start-based clear with explicit start()/stop() -- reintroducing an implicit restart here would silently reset elapsed_time/statistics too, not just the spectrum, so that trade-off is left to the caller.)
statistics.get_events_lost(): raised NotImplementedError as expected (IIO MCA backend: vdpp-pulse-processor.c (IP version 101) has no events-lost counter register -- throughput_error_counter is a different, separate statistic and would be misleading to alias here.)
sync.get_enable(): raised NotImplementedError as expected (IIO MCA backend: no sync-trigger core found in vdpp-pulse-processor.c or vdpp-input-filter.c -- this appears to be a separate, not-yet-ported device. SyncTrigger is unused by MCAController, so this is safe to leave unimplemented.)
sync.g

In [12]:
d.close()

## Notes / gotchas confirmed by this run

- **Real bug found and fixed: double-converted X2/X8/PLUS1_X8 fields.** See
  the intro for the full account. Affected every one of:
  `pretrigger_samples`, `frame_samples`, `cfd_delay`, both CFD/PSD
  time-window pairs, `charge_comparison_time`, `crrc2_fdelay`,
  `trapezoid_r`/`m`/`time` -- all now plain passthroughs, all re-confirmed
  live above, several at both ends of their valid range. The bug was
  symmetric (get-after-set looked correct) and only surfaced because a
  double-converted `crrc2_fdelay` value landed outside its valid range and
  the driver rejected it with a real `OSError` -- worth remembering next
  time a "round-trips fine locally" check feels like enough evidence that a
  register-level conversion is correct.
- **`debug_data`/`histogram_data` are not reachable over this IIO network
  context, confirmed multiple ways, not just one failed read.** Checked via
  `iio._d_attr_count()`, `_d_debug_attr_count()`, and `_d_buffer_attr_count()`
  directly (bypassing the Python `attrs` dict entirely) -- all three
  exclude both names, matching their absence from `iio_info`'s own
  device-attribute dump (57 attributes listed for `vdpp_pulse_processor`,
  neither of these two among them). Every other attribute on the same
  device works fine over this transport, including scale/available/counter
  attributes that share the same sysfs directory. This is consistent with
  a `bin_attribute`-vs-`device_attribute` discovery gap in this board's
  `iiod` build (`bin_attrs` in a kernel `attribute_group` may not be
  getting picked up by `iiod`'s local sysfs scan the way plain
  `device_attribute`s are) -- worth raising with whoever maintains the
  board image/`iiod` build, since nothing on the Python client side can
  route around an attribute `iiod` itself doesn't know exists.
- **`MCAWorker` needed a second fallback, not just the first one.** The
  original fix (falling back to the last-known histogram on `OSError`
  `EBUSY`) didn't cover this: `read_histogram()`/`read_waveform_banks()`
  raise `RuntimeError` here, not `OSError`, and `read_waveform_banks()` has
  no fallback at all originally -- so before this run, *every* tick would
  have aborted immediately (waveforms and statistics included), not just
  the spectrum. Confirmed live in the "Run a short measurement" cell above:
  `count_rate` visibly changes between prints throughout the run despite
  both `acquire_spectrum()`/`acquire_waveforms()` failing on every single
  call.
- **`trapez_T`/`trapez_E` really do map to `trapezoid_beta_raw`/
  `trapezoid_time`, confirmed by writing values only valid for one side.**
  See the CRRC2/trapezoid cell above.
- **`trapezoid_r`'s real minimum is `16`, confirmed via a raw `OSError`
  at `8`.** `mca.py`'s `MCAParam.TRAPEZ_R` `RangeSpec` still says `0` --
  worth fixing there too, since `MultiChannelAnalyzer.filters.trapezoid.
  set_R()` doesn't validate at all (no `MCA_PARAMETER_SPECS` check exists
  for filter sub-objects), so a value 0..15 reaches the driver unfiltered.
- **The LP preset labels are correct in the driver, `mca.py`'s comment
  isn't.** `MCAParam.LP_PRESET`'s comment says "0=200 MHz, 1=700 MHz,
  2=average"; the driver's real names are `"200mhz"`/`"70mhz"`/
  `"moving_average"`. Confirmed live: index 2's coefficients are all
  identical (`2730`), a genuine moving average, and index 1's coefficients
  are visibly different from index 0's -- just a stale comment, not a
  functional issue (the index values themselves already matched).
- **List-mode DMA (`vdpp-lm-frame`) is real hardware on this board, not
  ported to this backend yet.** `iio_info` shows `vdpp_lm_frame` (buffer
  capable) devices alongside `vdpp_pulse_processor`/`vdpp_input_filter` --
  `d.mca_dma` stays `None` and `set_dma_enable()` stays a no-op until that
  driver is reviewed and ported.